In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../../data/processed")
df = pd.read_parquet(DATA_DIR / "merged_clean.parquet")
print(f"Loaded: {df.shape}")
print(f"Fraud rate: {df['isFraud'].mean():.4f}")

Loaded: (590540, 425)
Fraud rate: 0.0350


In [2]:
# The cutoff from EDA — 80th percentile of TransactionDT
cutoff_dt = df["TransactionDT"].quantile(0.80)

train_mask = df["TransactionDT"] <= cutoff_dt
test_mask  = df["TransactionDT"] >  cutoff_dt

print(f"Train rows: {train_mask.sum():,}  (fraud rate: {df.loc[train_mask, 'isFraud'].mean():.4f})")
print(f"Test  rows: {test_mask.sum():,}  (fraud rate: {df.loc[test_mask, 'isFraud'].mean():.4f})")
print(f"\n⚠ All feature engineering stats will be computed ONLY from train_mask")

Train rows: 472,432  (fraud rate: 0.0351)
Test  rows: 118,108  (fraud rate: 0.0344)

⚠ All feature engineering stats will be computed ONLY from train_mask


In [3]:
# Hour as cyclical (so 23 and 0 are 'close' rather than far apart)
df["hour"] = (df["TransactionDT"] / 3600) % 24
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Day-of-dataset (linear, no cycle)
df["day_of_dataset"] = (df["TransactionDT"] / 86400).astype(int)

# Day-of-week (assume day 0 = Monday — we don't actually know, but the model will learn the pattern)
df["dow"] = df["day_of_dataset"] % 7

print("Time features added:")
print(df[["hour", "hour_sin", "hour_cos", "day_of_dataset", "dow"]].describe())



#A model treating hour as a plain number thinks hour=23 is 23 units away from hour=0. But in reality, 11 PM and midnight are adjacent. Sin/cos encoding maps them to nearby points on a circle.

Time features added:
                hour       hour_sin       hour_cos  day_of_dataset  \
count  590540.000000  590540.000000  590540.000000   590540.000000   
mean       14.363484      -0.340268       0.264983       84.729199   
std         7.618839       0.621251       0.654255       53.437277   
min         0.000000      -1.000000      -1.000000        1.000000   
25%         6.785625      -0.890808      -0.289171       35.000000   
50%        16.840000      -0.545188       0.449644       84.000000   
75%        20.408889       0.112986       0.873241      130.000000   
max        23.999722       1.000000       1.000000      182.000000   

                 dow  
count  590540.000000  
mean        2.957940  
std         2.033973  
min         0.000000  
25%         1.000000  
50%         3.000000  
75%         5.000000  
max         6.000000  


In [4]:
df["has_identity"] = df["DeviceType"].notna().astype(int)

# How many identity fields are populated for this transaction?
id_cols = [c for c in df.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
df["identity_field_count"] = df[id_cols].notna().sum(axis=1)

print(f"has_identity distribution:")
print(df["has_identity"].value_counts())
print(f"\nidentity_field_count by class:")
print(df.groupby("isFraud")["identity_field_count"].agg(["mean", "median", "std"]))

has_identity distribution:
has_identity
0    449730
1    140810
Name: count, dtype: int64

identity_field_count by class:
              mean  median        std
isFraud                              
0         5.784501     0.0  10.746539
1        13.435997    19.0  12.529348


In [5]:
# Log transformation (handles the right skew)
df["amount_log"] = np.log1p(df["TransactionAmt"])

# Decimal cents — fraud sometimes uses unusual cent values (e.g., $X.99 to test cards)
df["amount_cents"] = ((df["TransactionAmt"] - df["TransactionAmt"].astype(int)) * 100).round().astype(int)
df["amount_is_round"] = (df["TransactionAmt"] == df["TransactionAmt"].astype(int)).astype(int)

print("Amount features added.")
print(df[["amount_log", "amount_cents", "amount_is_round"]].describe())
print(f"\namount_is_round by class:")
print(df.groupby("isFraud")["amount_is_round"].mean())

Amount features added.
          amount_log   amount_cents  amount_is_round
count  590540.000000  590540.000000    590540.000000
mean        4.382960      37.944779         0.516498
std         0.937183      43.412080         0.499728
min         0.223943       0.000000         0.000000
25%         3.791459       0.000000         0.000000
50%         4.245190       0.000000         1.000000
75%         4.836282      95.000000         1.000000
max        10.371564     100.000000         1.000000

amount_is_round by class:
isFraud
0    0.516131
1    0.526642
Name: amount_is_round, dtype: float64


In [6]:
# Compute statistics on the TRAINING set only
train_df = df.loc[train_mask]

# Bayesian-smoothed card1 fraud rate
# Smoothing prior: assume 100 prior observations at the global rate
# This means cards with <100 training transactions stay anchored to global,
# and only well-observed cards get meaningful card-specific risk encoding
SMOOTHING_PRIOR = 100
global_fraud_rate = train_df["isFraud"].mean()

card1_stats = train_df.groupby("card1").agg(
    card1_amt_mean=("TransactionAmt", "mean"),
    card1_amt_std=("TransactionAmt", "std"),
    card1_txn_count=("TransactionAmt", "count"),
    card1_fraud_count=("isFraud", "sum"),
).reset_index()

# Bayesian smoothing
card1_stats["card1_fraud_rate"] = (
    (card1_stats["card1_fraud_count"] + SMOOTHING_PRIOR * global_fraud_rate)
    / (card1_stats["card1_txn_count"] + SMOOTHING_PRIOR)
)
card1_stats = card1_stats.drop(columns="card1_fraud_count")

print(f"card1 stats computed from {len(train_df):,} training rows")
print(f"Unique card1 values seen in train: {card1_stats.shape[0]:,}")
print(f"\nSmoothed card1_fraud_rate distribution:")
print(card1_stats["card1_fraud_rate"].describe())
print(f"\nGlobal fraud rate (anchor): {global_fraud_rate:.4f}")
print(f"  → Cards with <100 txns will have card1_fraud_rate close to this value")

# Map these statistics back onto the FULL dataset
df = df.merge(card1_stats, on="card1", how="left")

# Cold-start fill — unseen cards in test get the global rate
df["card1_fraud_rate"] = df["card1_fraud_rate"].fillna(global_fraud_rate)
df["card1_amt_mean"] = df["card1_amt_mean"].fillna(df.loc[train_mask, "TransactionAmt"].median())
df["card1_amt_std"] = df["card1_amt_std"].fillna(df.loc[train_mask, "TransactionAmt"].std())
df["card1_txn_count"] = df["card1_txn_count"].fillna(1)

test_unseen = (df.loc[test_mask, "card1_txn_count"] == 1).sum()
print(f"\nTest rows with unseen card1 (cold-start): {test_unseen:,} ({test_unseen / test_mask.sum():.2%})")

# Amount-relative features
df["amount_vs_card1_mean"] = df["TransactionAmt"] / df["card1_amt_mean"]
df["amount_vs_card1_z"] = (df["TransactionAmt"] - df["card1_amt_mean"]) / (df["card1_amt_std"] + 1e-6)

card1 stats computed from 472,432 training rows
Unique card1 values seen in train: 12,730

Smoothed card1_fraud_rate distribution:
count    12730.000000
mean         0.034712
std          0.014204
min          0.003013
25%          0.032234
50%          0.034112
75%          0.034787
max          0.339827
Name: card1_fraud_rate, dtype: float64

Global fraud rate (anchor): 0.0351
  → Cards with <100 txns will have card1_fraud_rate close to this value

Test rows with unseen card1 (cold-start): 2,275 (1.93%)


In [7]:
# Email domain risk encoding (computed from training only)
def get_top_domain(s):
    """Extract the top-level part: 'gmail.com' → 'gmail', 'yahoo.co.uk' → 'yahoo'"""
    if pd.isna(s):
        return "missing"
    return s.split(".")[0].lower()

for col in ["P_emaildomain", "R_emaildomain"]:
    df[f"{col}_top"] = df[col].apply(get_top_domain)

# Compute fraud rate per email domain from training data
for col in ["P_emaildomain_top", "R_emaildomain_top"]:
    domain_stats = train_df.assign(_top=df.loc[train_mask, col].values).groupby("_top")["isFraud"].agg(["mean", "count"])
    # Smoothing: domains with <50 transactions get the global mean (avoid overfitting on rare domains)
    global_rate = train_df["isFraud"].mean()
    domain_stats["smoothed"] = np.where(
        domain_stats["count"] >= 50,
        domain_stats["mean"],
        global_rate
    )
    
    # Map back
    mapping = domain_stats["smoothed"].to_dict()
    df[f"{col}_fraud_rate"] = df[col].map(mapping).fillna(global_rate)

print("Email domain features added.")
print("\nTop 10 risky P_emaildomains by training-set fraud rate:")
print(train_df.assign(_top=df.loc[train_mask, "P_emaildomain_top"].values)
      .groupby("_top")["isFraud"].agg(["mean", "count"])
      .query("count >= 50")
      .sort_values("mean", ascending=False)
      .head(10))



#if a domain has fewer than 50 transactions in training, we use the global fraud rate instead of its computed rate. Without smoothing, a domain with 2 transactions both happening to be fraud gets fraud_rate = 1.0, which is meaningless overfitting

Email domain features added.

Top 10 risky P_emaildomains by training-set fraud rate:
                 mean   count
_top                         
protonmail   0.462687      67
mail         0.194690     452
aim          0.149254     268
outlook      0.094907    4457
hotmail      0.051648   37736
gmail        0.043948  183103
suddenlink   0.035088     114
embarqmail   0.033333     210
frontiernet  0.032680     153
icloud       0.031803    5031


In [8]:
# Frequency encoding: replace each value with how often it appears in training
for col in ["card1", "addr1", "card2", "card5"]:
    if col in df.columns:
        freq_map = train_df[col].value_counts().to_dict()
        df[f"{col}_freq"] = df[col].map(freq_map).fillna(0)

print("Frequency encoding added for card1, addr1, card2, card5.")
print(df[[c for c in df.columns if c.endswith("_freq")]].describe())

Frequency encoding added for card1, addr1, card2, card5.
          card1_freq     addr1_freq     card2_freq     card5_freq
count  590540.000000  590540.000000  590540.000000  590540.000000
mean     2027.677185   15909.921792   14357.724268  136046.133693
std      2978.955503   12937.067092   14201.195105  102930.760166
min         0.000000       0.000000       0.000000       0.000000
25%       105.000000    4466.000000    2148.000000   22324.000000
50%       726.000000   12211.000000    7769.000000  237195.000000
75%      2524.000000   31758.000000   30168.000000  237195.000000
max     12205.000000   37304.000000   38743.000000  237195.000000


In [9]:
# Drop the now-redundant original columns
DROP_FINAL = ["P_emaildomain_top", "R_emaildomain_top"]
df_final = df.drop(columns=DROP_FINAL)

# Sanity check: any features that are 100% NaN?
all_nan = df_final.columns[df_final.isna().all()].tolist()
if all_nan:
    print(f"⚠ Dropping {len(all_nan)} all-NaN columns: {all_nan}")
    df_final = df_final.drop(columns=all_nan)

print(f"\nFinal shape: {df_final.shape}")
print(f"Features added: {df_final.shape[1] - 425}")

# Save
out_path = Path("../../data/processed")
df_final.to_parquet(out_path / "features_ready.parquet", index=False)
print(f"\nSaved: {out_path / 'features_ready.parquet'}")
print(f"Size: {(out_path / 'features_ready.parquet').stat().st_size / 1e6:.1f} MB")


Final shape: (590540, 445)
Features added: 20

Saved: ../../data/processed/features_ready.parquet
Size: 110.5 MB


In [10]:
# Re-derive masks on the final df (in case row order shifted from merges)
train_mask_final = df_final["TransactionDT"] <= cutoff_dt
test_mask_final  = df_final["TransactionDT"] >  cutoff_dt

print(f"Train: {train_mask_final.sum():,} rows, fraud rate {df_final.loc[train_mask_final, 'isFraud'].mean():.4f}")
print(f"Test:  {test_mask_final.sum():,} rows, fraud rate {df_final.loc[test_mask_final, 'isFraud'].mean():.4f}")

# Confirm no leakage by checking a derived feature
# card1_amt_mean for the test set should match what we got from training
sample_card1 = df_final.loc[test_mask_final, "card1"].iloc[0]
expected = train_df[train_df["card1"] == sample_card1]["TransactionAmt"].mean()
actual = df_final.loc[(df_final["card1"] == sample_card1) & test_mask_final, "card1_amt_mean"].iloc[0]
print(f"\nLeakage check: card1={sample_card1}")
print(f"  Expected card1_amt_mean (computed from train): {expected:.2f}")
print(f"  Actual value in test feature: {actual:.2f}")
print(f"  Match: {np.isclose(expected, actual)}")

Train: 472,432 rows, fraud rate 0.0351
Test:  118,108 rows, fraud rate 0.0344

Leakage check: card1=9300
  Expected card1_amt_mean (computed from train): 41.26
  Actual value in test feature: 41.26
  Match: True
